# Concepts used in week 3 (Notes Sandro)
- Decision Trees
  - Simple Tree
  - Random Forrest
  - XGBoost
- Over/Underfitting
- Parameters vs. Hyperparameters
- Difference Train / Test / Validation Set
- Cross-Validaiton & Bootstrapping
- Scaling
- Grid Search
- SHAP Values
- Regularization Methods (Ridge (L2), Lasso (L1, kills unnecessary parameters)

# Notes on features
- Days since last hba1c seems arbitrary (depending on when you went for check). Might not be a good indicator.
- Why not look at Average / Median Visit interval or similar.

# For "free" excercise:
We can change all of these steps. (also outliert detection etc)
1. CSVs + Notes
2. Create combined csvs
3. Feature Engineering
4. Model Selection
5. Explanations (Feature Importance etc.)

e.g LSTM could be interesting ->
how would we transform our problem to LSTM problem?

-> Create Date Vectors with 22 Dimensiones (22x1), because we have 22 features
-> What pytorch expects:

-> LSTM
-> TabPFN https://github.com/PriorLabs/TabPFN

# Week 3 Homework: End-to-End Clinical Decision Support Pipeline

In Weeks 1-3, you built the individual pieces:
- **Week 1**: OCR cleaning, clinical note extraction
- **Week 2**: Patient profiles, 22-feature engineering with TDD
- **Week 3**: XGBoost tuning, SHAP interpretability, robustness

Now you **wire them into a system** — a single `EHRPipeline` class that goes from raw Synthea CSVs to clinician-readable risk predictions with full SHAP explainability.

---

## What You'll Build

| Part | What You Wire Up | Week Origin |
|------|-----------------|-------------|
| 1 | OCR cleaning + note extraction | Week 1 |
| 2 | Patient profile construction | Week 2 |
| 3 | Feature computation at cutoff date | Week 2 |
| 4 | Model loading + prediction | Week 3 Session 2 |
| 5 | Global SHAP analysis | Week 3 Session 3 |
| 6 | Individual explanations + clinical translation | Week 3 Session 3 |
| 7 | Integration test | All weeks |
| 8 | Evaluation + deployment readiness | Week 2 S3 + Week 3 S4 |

**Key principle**: You import what you already built — no reimplementation. The new work is composing, connecting, and adding the prediction/explanation layer.

---

## Setup

## Setting Environment Up for Colab

Mount Google Drive and set the repository path so this notebook can access the EHR data and source code.

In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


### Project Configuration

This notebook loads paths from a YAML config file instead of hard-coded paths.

**Setup (one-time):**
1. In Colab's left sidebar, click the **Key** icon (Secrets)
2. Add a secret named `PROJECT_CONFIG_PATH`
3. Set the value to your config file path (e.g., `/content/drive/MyDrive/Project/config.yaml`)
4. Toggle "Notebook access" ON

In [2]:
from google.colab import userdata
import yaml

try:
    config_path = userdata.get('PROJECT_CONFIG_PATH')
except:
    config_path = input("Enter path to your config.yaml: ")

with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

REPO_PATH = config['repo_path']
print(f"Repository path: {REPO_PATH}")

Repository path: /content/drive/MyDrive/MASAID/AI_Project_Group/repo


### Install Source Code as Package

`pip install` installs the local source code as a Python package so you can import directly from it (e.g., `from ehr_pipeline import EHRPipeline`). Re-run this cell after making changes to the source code.

In [3]:
!pip install {REPO_PATH}/src/ shap xgboost -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [4]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print("Core libraries imported")

Core libraries imported


In [5]:
import shap
import xgboost as xgb
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve

def get_shap_values(raw):
    """Handle both list (multiclass) and array (binary) SHAP outputs."""
    if isinstance(raw, list):
        return raw[1] if len(raw) > 1 else raw[0]
    return raw

print("ML and SHAP libraries imported")

ML and SHAP libraries imported


In [6]:
DATA_DIR = os.path.join(REPO_PATH, 'data', 'week_1', 'processed_data', 'csv')
NOTES_FILE = os.path.join(REPO_PATH, 'data', 'week_1', 'processed_data', 'notes_for_extraction.csv')
MODEL_PATH = os.path.join(REPO_PATH, 'data', 'week_3', 'best_xgboost_model.pkl')
WEEK2_DATA = os.path.join(REPO_PATH, 'data', 'week_2')
OUTPUT_DIR = os.path.join(REPO_PATH, 'data', 'week_3')
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"REPO_PATH: {REPO_PATH}")
print(f"DATA_DIR:     {DATA_DIR}")
print(f"MODEL_PATH:   {MODEL_PATH}")
print(f"NOTES FILE:   {NOTES_FILE}")

REPO_PATH: /content/drive/MyDrive/MASAID/AI_Project_Group/repo
DATA_DIR:     /content/drive/MyDrive/MASAID/AI_Project_Group/repo/data/week_1/processed_data/csv
MODEL_PATH:   /content/drive/MyDrive/MASAID/AI_Project_Group/repo/data/week_3/best_xgboost_model.pkl
NOTES FILE:   /content/drive/MyDrive/MASAID/AI_Project_Group/repo/data/week_1/processed_data/notes_for_extraction.csv


In [7]:
# After pip install, ehr_pipeline is importable directly.
# WORKFLOW: Edit src/ehr_pipeline.py to implement the TODOs,
# then re-run the pip install cell above to pick up your changes.
from ehr_pipeline import EHRPipeline, print_explanation, CLINICAL_MEANINGS

print(f"EHRPipeline imported")
print(f"Clinical meanings defined for {len(CLINICAL_MEANINGS)} features")

EHRPipeline imported
Clinical meanings defined for 22 features


---

# Part 1: Data Loading & Cleaning

The pipeline's first three methods compose Week 1 building blocks:

| Method | What it calls from Week 1 | Purpose |
|--------|--------------------------|----------|
| `load_csvs()` | `pd.read_csv()` | Read Synthea CSVs |
| `clean_notes()` | `strip_html()`, `fix_ocr_text()` | Clean clinical notes |
| `extract_notes()` | `extract_all_from_note()` | Extract vitals/labs/meds from text |

All three return `self` for method chaining.

In [8]:
# Create pipeline instance -- loads the pre-trained XGBoost model
pipe = EHRPipeline(
    data_dir=DATA_DIR,
    model_path=MODEL_PATH,
    notes_file=NOTES_FILE,
)

# Stage 1: Load Synthea CSVs
pipe.load_csvs()

# Verify row counts
print(f"\nVerification:")
print(f"  Patients:      {len(pipe.patients_df):,}")
print(f"  Encounters:    {len(pipe.encounters_df):,}")
print(f"  Conditions:    {len(pipe.conditions_df):,}")
print(f"  Observations:  {len(pipe.observations_df):,}")
print(f"  Medications:   {len(pipe.medications_df):,}")
if pipe.notes_df is not None:
    print(f"  Notes:         {len(pipe.notes_df):,}")

Loaded model from best_xgboost_model.pkl

Verification:
  Patients:      9,998
  Encounters:    73,544
  Conditions:    31,589
  Observations:  352,634
  Medications:   40,177
  Notes:         15,540


In [9]:
for name in pipe._CSV_NAMES:
    df = getattr(pipe, f"{name}_df")
    print(f"{name}_df: {len(df) if df is not None else 'None'}")

patients_df: 9998
encounters_df: 73544
conditions_df: 31589
observations_df: 352634
medications_df: 40177


In [10]:
# Stage 2-3: Clean notes and extract structured data
if pipe.notes_df is not None:
    obs_before = len(pipe.observations_df)
    sample_note = pipe.notes_df['note_text'].iloc[0][:200]
    print(f"Sample note BEFORE cleaning:\n  {sample_note}...\n")

    pipe.clean_notes().extract_notes()

    sample_note_after = pipe.notes_df['note_text'].iloc[0][:200]
    print(f"\nSample note AFTER cleaning:\n  {sample_note_after}...")

    obs_after = len(pipe.observations_df)
    print(f"\nObservations grew: {obs_before:,} -> {obs_after:,} "
          f"(+{obs_after - obs_before:,} from note extraction)")
else:
    print("No notes file \u2014 skipping clean_notes() and extract_notes()")

Sample note BEFORE cleaning:
  **Patient : ** 76-year-old F **Encounter Type : ** Outpatient **Date : ** 2012-09-30 --- **hx of Present Illness : ** 76-year-old female returns for routine follow-up . She last visited the clinic thr...


Sample note AFTER cleaning:
  **Patient : ** 76-year-old F **Encounter Type : ** Outpatient **Date : ** 2012-09-30 --- **hx of Present Illness : ** 76-year-old female returns for routine follow-up . She last visited the clinic thr...

Observations grew: 352,634 -> 396,976 (+44,342 from note extraction)


---

# Part 2: Profile Building

`build_profiles()` calls `ExtendedPatientProfile.load_from_dataframes()` — the same method you used in Week 2 to construct per-patient objects with sorted timelines (HbA1c, BP, eGFR, medications, encounters, etc.).

In [11]:
pipe.build_profiles()

# Verify: expect ~2,600 diabetic profiles
print(f"Diabetic profiles: {len(pipe.profiles):,}")

# Inspect a sample patient's timelines
sample_id = list(pipe.profiles.keys())[0]
profile = pipe.profiles[sample_id]

print(f"\nSample patient: {sample_id}")
print(f"  HbA1c readings:       {len(profile.hba1c_timeline)}")
print(f"  BP readings:          {len(profile.bp_timeline)}")
print(f"  eGFR readings:        {len(profile.egfr_timeline)}")
print(f"  Encounters:           {len(profile.encounter_timeline)}")
print(f"  Medications:          {len(profile.medication_timeline)}")
print(f"  Emergency visits:     {len(profile.emergency_visit_timeline)}")
print(f"  Hospitalizations:     {len(profile.hospitalization_timeline)}")
print(f"  Complications:        {len(profile.complication_timeline)}")
print(f"  Care gaps:            {len(profile.care_gap_timeline)}")

Loading 2,621 patient profiles...


Building profiles: 100%|██████████| 2621/2621 [01:47<00:00, 24.34it/s]


✅ Loaded 2,621 profiles
   With encounters: 2,621
   With ≥60 days data: 2,109 (will generate instances)
Diabetic profiles: 2,621

Sample patient: fce0fcd4-a85c-4a5b-b926-d2ef97ee9a6c
  HbA1c readings:       14
  BP readings:          10
  eGFR readings:        7
  Encounters:           11
  Medications:          2
  Emergency visits:     0
  Hospitalizations:     0
  Complications:        0
  Care gaps:            6


---

# Part 3: Single Patient Features

`get_patient_features(patient_id, date)` calls the profile's `get_daily_features(date)` method, which computes all 22 features using **only data on or before the cutoff date** (temporal safety).

We verify against the Week 2 Session 1 TDD sample patient to confirm continuity.

In [12]:
# Week 2 TDD sample patient and cutoff date
SAMPLE_PATIENT = "21a69c84-ff08-4928-9e96-95c8aac45a6f"
CUTOFF_DATE = "2012-07-05"

if SAMPLE_PATIENT in pipe.profiles:
    features = pipe.get_patient_features(SAMPLE_PATIENT, CUTOFF_DATE)

    print(f"Features for patient {SAMPLE_PATIENT[:12]}... at {CUTOFF_DATE}:")
    print("=" * 60)
    for name, value in features.items():
        if value is None or (isinstance(value, float) and pd.isna(value)):
            val_str = "not measured"
        elif isinstance(value, (int, float)):
            val_str = f"{value:.2f}"
        else:
            val_str = str(value)
        print(f"  {name:<35} = {val_str}")
    print(f"\nTotal features: {len(features)}")
else:
    print(f"Sample patient not found \u2014 using a different patient")
    alt_id = list(pipe.profiles.keys())[0]
    features = pipe.get_patient_features(alt_id, "2015-01-01")
    print(f"Features for {alt_id[:12]}... ({len(features)} features)")

Features for patient 21a69c84-ff0... at 2012-07-05:
  age_at_date                         = 39.00
  days_since_last_hba1c               = 19.00
  days_since_last_encounter           = 3.00
  days_since_medication_change        = 5.00
  days_since_emergency_visit          = not measured
  days_since_hospitalization          = not measured
  encounters_last_90d                 = 8.00
  emergency_visits_last_180d          = 0.00
  hospitalizations_last_365d          = 0.00
  medication_changes_last_90d         = 1.00
  current_hba1c_level                 = 12.70
  hba1c_trend_last_180d               = 1.00
  diabetes_complication_count         = 0.00
  care_gaps_count                     = 1.00
  longest_care_gap_days               = 243.00
  current_systolic_bp                 = 167.00
  current_diastolic_bp                = 107.00
  bp_trend_last_180d                  = 1.00
  current_egfr                        = 64.20
  egfr_trend_last_365d                = 0.00
  active_medication_co

**TDD Continuity Check**: The feature values above should match what you computed in Week 2 Session 1. The pipeline calls the exact same `ExtendedPatientProfile` methods \u2014 just wired through a single interface.

---

# Part 4: Prediction

`predict_patient(patient_id, date)` chains:
1. `get_patient_features()` \u2014 compute 22 features at cutoff date
2. Build a DataFrame row (NaN passthrough \u2014 XGBoost handles missing values natively)
3. `model.predict_proba()` \u2014 return the probability of a high-risk event in the next 30 days

The model was trained in Session 2 and loaded from pickle \u2014 we do NOT retrain.

In [13]:
# Predict on the TDD sample patient
if SAMPLE_PATIENT in pipe.profiles:
    prob = pipe.predict_patient(SAMPLE_PATIENT, CUTOFF_DATE)
    print(f"Patient: {SAMPLE_PATIENT[:12]}...")
    print(f"Date:    {CUTOFF_DATE}")
    print(f"Risk:    {prob:.1%}")
    print(f"\nProbability is in [0,1]: {0 <= prob <= 1}")

Patient: 21a69c84-ff0...
Date:    2012-07-05
Risk:    99.2%

Probability is in [0,1]: True


In [14]:
# Predict on several patients at different dates
sample_patients = list(pipe.profiles.keys())[:10]
test_date = "2015-06-01"

print(f"Predictions at {test_date}:")
print("-" * 60)
for pid in sample_patients:
    try:
        prob = pipe.predict_patient(pid, test_date)
        level = "CRITICAL" if prob >= 0.8 else "HIGH" if prob >= 0.6 else "MODERATE" if prob >= 0.4 else "LOW"
        print(f"  {pid[:20]}...  {prob:6.1%}  [{level}]")
    except (KeyError, ValueError) as e:
        print(f"  {pid[:20]}...  skipped ({e})")

Predictions at 2015-06-01:
------------------------------------------------------------
  fce0fcd4-a85c-4a5b-b...   17.5%  [LOW]
  efb5fdb4-ce87-470f-a...   33.7%  [LOW]
  60cda45c-843b-428e-9...   55.5%  [MODERATE]
  f5f3fcaa-c1dd-4d04-9...    4.8%  [LOW]
  efb98d06-ef85-4288-9...    8.4%  [LOW]
  536b5b55-be54-41b3-8...   23.6%  [LOW]
  97a7a977-5bd0-4b3b-9...   27.3%  [LOW]
  4f34036b-59dd-4ed5-b...    9.1%  [LOW]
  075d25ec-0ff3-48f6-8...    6.6%  [LOW]
  8066867b-bec6-49cc-9...   11.4%  [LOW]


---

# Part 5: Global SHAP Analysis

We use **SHAP (SHapley Additive exPlanations)** \u2014 the gold standard for model interpretability in clinical settings \u2014 to understand which features drive predictions **across all patients**.

### SHAP Key Concepts

| Concept | Meaning | Example |
|---------|---------|---------|
| **Base value** | Average prediction across all patients | ~3% (reflects the low positive rate in our dataset) |
| **SHAP value** | Contribution of one feature | +0.8 from high HbA1c |
| **Positive SHAP** | Feature increases risk | Red in plots |
| **Negative SHAP** | Feature decreases risk | Blue in plots |
| **Mean \|SHAP\|** | Average impact magnitude | Feature ranking metric |

## 5a. Setup: Load Data & Create SHAP Explainer

For batch SHAP analysis we need the pre-computed feature matrix from Week 2 \u2014 calling `pipe.build_feature_matrix()` would regenerate ~580K instances from scratch, which is slow. Instead we load the saved CSV directly.

**NaN Passthrough**: Both XGBoost and TreeExplainer handle NaN natively. Using `fillna(0)` would create clinically impossible values (e.g., eGFR=0) and distort SHAP explanations.

---

<details>
<summary><strong>Hint 1 — SHAP TreeExplainer fix</strong> (click to expand)</summary>

```python
def _safe_float(x):
    if isinstance(x, str) and x.startswith('[') and x.endswith(']'):
        return _builtin_float(x.strip('[]'))
    return _builtin_float(x)

builtins.float = _safe_float
explainer = shap.TreeExplainer(pipe.model)
builtins.float = _builtin_float
```
</details>

In [15]:
# Load pre-computed training data from Week 2
# (faster than regenerating ~580K instances via pipe.build_feature_matrix())
classifier_data = pd.read_csv(
    os.path.join(WEEK2_DATA, 'classifier_training_data_22_features.csv'),
    low_memory=False
)

label_col = 'will_have_high_risk_event_next_30d'
patient_col = 'patient_id'

# Convert bmi_category from string to numeric if needed
if 'bmi_category' in classifier_data.columns and classifier_data['bmi_category'].dtype == 'object':
    bmi_mapping = {'underweight': 1, 'normal': 2, 'overweight': 3, 'obese': 4}
    classifier_data['bmi_category'] = classifier_data['bmi_category'].map(bmi_mapping)

# Drop rows with NaN labels
valid_mask = classifier_data[label_col].notna()
classifier_data = classifier_data[valid_mask].copy()
classifier_data[label_col] = classifier_data[label_col].astype(int)

# Identify feature columns
exclude_patterns = ['label', 'risk', 'target', 'will_have', 'next', 'survival']
feature_cols = [c for c in classifier_data.columns
                if c not in ['patient_id', 'date']
                and not any(p in c.lower() for p in exclude_patterns)]

print(f"Loaded {len(classifier_data):,} instances, {classifier_data['patient_id'].nunique():,} patients")
print(f"Features: {len(feature_cols)}, Positive rate: {classifier_data[label_col].mean():.2%}")

Loaded 579,356 instances, 2,109 patients
Features: 22, Positive rate: 2.80%


In [16]:
# Patient-level train/test split (same as Session 2/3)
X = classifier_data[feature_cols]
y = classifier_data[label_col]
groups = classifier_data[patient_col]

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

# Predict on test set
y_prob = pipe.model.predict_proba(X_test)[:, 1]

print(f"Train: {len(X_train):,} instances")
print(f"Test:  {len(X_test):,} instances")
print(f"Test ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}")
print(f"Test PR-AUC:  {average_precision_score(y_test, y_prob):.4f}")

Train: 466,059 instances
Test:  113,297 instances
Test ROC-AUC: 0.9544
Test PR-AUC:  0.8783


In [17]:
# Create SHAP TreeExplainer
# HOMEWORK TODO: Create a SHAP TreeExplainer for the loaded XGBoost model.
#
# Problem: SHAP 0.49 with XGBoost >= 2.0 stores base_score as '[5E-1]'
# which causes a ValueError in TreeExplainer. You need to temporarily
# monkey-patch builtins.float to handle this format.
#
# Steps:
#   1. Save the original builtins.float
#   2. Define _safe_float(x) that strips '[]' from string inputs
#   3. Replace builtins.float temporarily
#   4. Create shap.TreeExplainer(pipe.model)
#   5. Restore builtins.float immediately
#   6. Extract the expected_value (base value) for force plots

import builtins
_builtin_float = builtins.float

# TODO: Define _safe_float and create explainer
explainer = None

# TODO: Extract base value (scalar)
exp_val = None
base_prob = None

# TODO: Compute SHAP values on a sample of test data
n_samples = min(500, len(X_test))
X_test_sample = X_test.sample(n=n_samples, random_state=42)

print(f"Computing SHAP values for {n_samples} samples...")
shap_values_raw = None  # explainer.shap_values(X_test_sample)
shap_values = None      # get_shap_values(shap_values_raw)
print(f"SHAP values shape: {shap_values.shape if shap_values is not None else 'N/A'}")

Computing SHAP values for 500 samples...
SHAP values shape: N/A


## 5b. Global Feature Importance

Global importance answers: **"Which features matter most across all patients?"**

We compute the **mean absolute SHAP value** for each feature. We use absolute values because a feature with SHAP = +0.5 for some patients and -0.5 for others still has high importance \u2014 positive and negative contributions shouldn't cancel out.

---

<details>
<summary><strong>Hint 1 — Mean |SHAP| computation</strong> (click to expand)</summary>

`mean_abs_shap = np.abs(shap_values).mean(axis=0)` — one value per feature, representing average absolute impact.
</details>

In [18]:
# HOMEWORK TODO: Compute global feature importance using mean |SHAP|
#   1. Take absolute values of shap_values
#   2. Compute mean across axis=0 (all patients)
#   3. Build DataFrame with 'feature' and 'mean_abs_shap' columns
#   4. Sort descending by importance
#   5. Print top 15 features with a bar visualization
mean_abs_shap = None

feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'mean_abs_shap': mean_abs_shap
}).sort_values('mean_abs_shap', ascending=False)

print("Global Feature Importance (Mean |SHAP|):")
print("=" * 60)
for rank, (_, row) in enumerate(feature_importance.head(15).iterrows(), 1):
    bar = "\u2588" * int(row['mean_abs_shap'] / (mean_abs_shap.max() if mean_abs_shap is not None else 1) * 30)
    print(f"  {rank:2d}. {row['feature']:<35} {row['mean_abs_shap']:.4f}  {bar}")

Global Feature Importance (Mean |SHAP|):


TypeError: unsupported operand type(s) for /: 'NoneType' and 'int'

### SHAP Summary Plot (Beeswarm)

The beeswarm plot is the most information-dense SHAP visualization:

| Element | Meaning |
|---------|---------|
| **Y-axis** | Features ranked by importance (top = most important) |
| **X-axis** | SHAP value (left = decreases risk, right = increases risk) |
| **Each dot** | One patient's SHAP value for that feature |
| **Color** | Feature value (red = high, blue = low) |

**Clinical Validation**: If `current_hba1c_level` shows red dots (high HbA1c) on the right (increases risk), the model learned the correct clinical relationship.

In [ ]:
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_test_sample, max_display=15, show=False)
plt.title("SHAP Summary Plot \u2014 How Features Affect Risk Predictions")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test_sample, plot_type="bar", max_display=15, show=False)
plt.title("Mean |SHAP| \u2014 Global Feature Importance")
plt.tight_layout()
plt.show()

### Feature Dependence Plots

Dependence plots show **how a feature's value affects its SHAP contribution**:

| Pattern | Meaning | Example |
|---------|---------|---------|
| **Upward slope** | Higher values \u2192 higher risk | HbA1c: more = worse |
| **Downward slope** | Higher values \u2192 lower risk | eGFR: more = better kidney function |
| **Threshold effect** | Risk jumps at specific value | HbA1c > 7.0 triggers concern |
| **Color separation** | Interaction with another feature | HbA1c effect differs by age |

In [ ]:
top_features = feature_importance['feature'].head(4).tolist()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, feature in enumerate(top_features):
    shap.dependence_plot(feature, shap_values, X_test_sample, ax=axes[i], show=False)
    axes[i].set_title(f"Dependence: {feature[:35]}")

plt.tight_layout()
plt.show()

print("Top 4 features shown:")
for f in top_features:
    print(f"  - {f}: {CLINICAL_MEANINGS.get(f, '')[:60]}")

---

# Part 6: Individual Explanations & Clinical Translation

Part 5 showed what matters **across all patients**. Now we explain **specific patients** \u2014 the core clinical use case. When a nurse sees a high-risk alert, they need to know:

1. **Why is this patient flagged?** \u2192 Force plot + contributing factors
2. **What can we do about it?** \u2192 Clinical interpretation + suggested interventions
3. **Is this alert trustworthy?** \u2192 Compare TP, FN, FP explanations

## 6a. Force Plots

Force plots visualize individual predictions as a "tug of war" between features:

| Element | Meaning |
|---------|---------|
| **Base value** | Starting point (average prediction) |
| **Red arrows** | Features pushing prediction **higher** (increasing risk) |
| **Blue arrows** | Features pushing prediction **lower** (decreasing risk) |
| **Arrow width** | Magnitude of contribution |

**Clinical Analogy**: Red factors are warning signs; blue factors are protective elements keeping the risk from being even higher.

In [ ]:
shap.initjs()

# Identify high-risk patients from different patients
high_risk_threshold = 0.7
high_risk_mask = y_prob > high_risk_threshold
high_risk_all = np.where(high_risk_mask)[0]

# Deduplicate by patient_id, pick the 3 highest-risk
high_risk_df = pd.DataFrame({
    'idx': high_risk_all,
    'patient_id': classifier_data.iloc[test_idx[high_risk_all]][patient_col].values,
    'risk': y_prob[high_risk_all]
})
sample_high_risk = high_risk_df.drop_duplicates('patient_id').nlargest(3, 'risk')['idx'].values

print(f"Patients with risk > {high_risk_threshold:.0%}: {len(high_risk_all):,}")
print(f"Selected {len(sample_high_risk)} high-risk patients for detailed explanation")

## 6b. Natural Language Clinical Explanations

The functions below are for **data scientist analysis** \u2014 they take raw SHAP arrays and produce detailed reports showing both `contribution_pct` and raw SHAP values. For **production use**, the pipeline wraps this into a simpler API: `pipe.explain_patient()` + `print_explanation()` (demonstrated in Part 7).

| Metric | Who sees it | Purpose |
|--------|-------------|---------|
| **contribution_pct** | Clinicians | "HbA1c accounts for 35% of this patient's risk" |
| **Raw SHAP value** | Data scientists | Exact log-odds contribution for debugging |

---

<details>
<summary><strong>Hint 1 — contribution_pct</strong> (click to expand)</summary>

`contribution_pct = |shap_i| / sum(|shap_all|) * 100` — converts raw SHAP into a percentage budget. Use `explanation_df[explanation_df['shap'] > 0].nlargest(top_n, 'shap')` for risk factors.
</details>

<details>
<summary><strong>Hint 2 — Risk levels</strong> (click to expand)</summary>

CRITICAL >= 0.8, HIGH >= 0.6, MODERATE >= 0.4, else LOW
</details>

In [ ]:
# HOMEWORK TODO: Implement two functions for patient explanation.
#
# Function 1: generate_patient_explanation
#   Takes raw SHAP values and produces a structured dict with:
#   - contribution_pct: |shap_i| / sum(|shap|) * 100
#   - risk_factors: top-N features pushing risk UP (shap > 0)
#   - protective_factors: top-N features pushing risk DOWN (shap < 0)
#   - risk_level: CRITICAL/HIGH/MODERATE/LOW based on risk_score
#
# Function 2: print_clinical_explanation
#   Pretty-prints the explanation dict with clinical meanings

def generate_patient_explanation(features, shap_vals, feature_names, risk_score, top_n=5):
    """Generate structured explanation from SHAP values with contribution_pct.

    HOMEWORK TODO: Implement this function.

    Hint: Use np.abs() for absolute values, pd.DataFrame for structuring,
    .nlargest() and .nsmallest() for selecting top factors.
    """
    shap_flat = shap_vals.flatten()

    # TODO: Compute total absolute SHAP
    total_abs_shap = None

    # TODO: Build DataFrame with feature, value, shap, contribution_pct
    explanation_df = None

    # TODO: Split into risk_factors (shap > 0) and protective_factors (shap < 0)
    risk_factors = None
    protective_factors = None

    # TODO: Determine risk level
    risk_level = None

    return {
        'risk_score': risk_score,
        'risk_level': risk_level,
        'risk_factors': risk_factors,
        'protective_factors': protective_factors,
        'all_factors': explanation_df
    }


def print_clinical_explanation(explanation, patient_id="Unknown"):
    """Print formatted clinical explanation with contribution_pct and SHAP values.

    HOMEWORK TODO: Implement this function.

    Hint: Loop through explanation['risk_factors'].iterrows() and
    explanation['protective_factors'].iterrows(). Use CLINICAL_MEANINGS dict.
    """
    print("\n" + "=" * 70)
    print("PATIENT RISK ASSESSMENT")
    print("=" * 70)
    print(f"Patient ID: {patient_id}")
    print(f"Risk Score: {explanation['risk_score']:.1%}")
    print(f"Risk Level: {explanation['risk_level']}")

    # TODO: Print RISK-INCREASING FACTORS
    print("\n" + "-" * 70)
    print("RISK-INCREASING FACTORS:")
    print("-" * 70)
    # Loop through explanation['risk_factors'] and print each

    # TODO: Print PROTECTIVE FACTORS
    print("\n" + "-" * 70)
    print("PROTECTIVE FACTORS:")
    print("-" * 70)
    # Loop through explanation['protective_factors'] and print each

    print("\n" + "=" * 70)

print("Explanation functions defined")

In [ ]:
# Force plot + clinical explanation for one high-risk patient
idx = sample_high_risk[0]
patient_features = X_test.iloc[[idx]]
patient_shap_raw = explainer.shap_values(patient_features)
patient_shap = get_shap_values(patient_shap_raw)

patient_id = classifier_data.iloc[test_idx[idx]][patient_col]
actual = y_test.iloc[idx]

print(f"{'='*70}")
print(f"Patient: {patient_id[:20]}...")
print(f"Risk Score: {y_prob[idx]:.1%}  |  Actual: {'HIGH-RISK EVENT' if actual else 'No event'}")
print(f"{'='*70}")

# Force plot
shap.force_plot(
    float(exp_val),
    patient_shap[0],
    patient_features.iloc[0],
    matplotlib=True,
    show=False
)
plt.title(f"Force Plot -- Patient Risk: {y_prob[idx]:.1%}", fontsize=12, pad=20)
plt.tight_layout()
plt.show()

# Clinical explanation
explanation = generate_patient_explanation(
    patient_features, patient_shap, feature_cols, y_prob[idx]
)
print_clinical_explanation(explanation, patient_id=patient_id[:20] + "...")

## 6c. Clinical Interpretation Reference

This reference maps top features to clinical interpretations and **suggested interventions**. This artifact is essential for:
- Training clinical staff on interpreting model outputs
- Translating alerts into actionable care plans
- Regulatory compliance documentation (EU AI Act, GDPR Article 22)

---

<details>
<summary><strong>Hint 1 — Keyword matching pattern</strong> (click to expand)</summary>

```python
interventions_map = {
    'hba1c': ("Glycemic control marker", "Medication review"),
    'days_since': ("Care continuity indicator", "Schedule follow-up"),
    # ... more keywords
}
for key, (interp, interv) in interventions_map.items():
    if key in feat.lower():
        ...
```
</details>

In [ ]:
# HOMEWORK TODO: Build clinical interpretation table for top features
#   Map each feature to a clinical interpretation and suggested intervention.
#   Use keyword matching on feature names.
top_10_features = feature_importance.head(10).copy()

# TODO: Create an interventions_map dict mapping keywords to (interpretation, intervention) tuples
#   Example: 'hba1c' -> ("Glycemic control marker", "Medication review, diabetes education referral")
#   Cover: days_since, gap, hba1c, emergency, hospital, medication, complication,
#          encounter, egfr, bp, systolic, diastolic, age, bmi
interventions_map = {
    # TODO: Fill in keyword -> (interpretation, intervention) pairs
}

interpretations, interventions = [], []
for feat in top_10_features['feature']:
    matched = False
    for key, (interp, interv) in interventions_map.items():
        if key in feat.lower():
            interpretations.append(interp)
            interventions.append(interv)
            matched = True
            break
    if not matched:
        interpretations.append("Clinical risk indicator")
        interventions.append("Clinical review recommended")

top_10_features['interpretation'] = interpretations
top_10_features['intervention'] = interventions

print("CLINICAL INTERPRETATION REFERENCE")
print("=" * 110)
print(f"{'Rank':<5} {'Feature':<35} {'Mean |SHAP|':<12} {'Interpretation':<30} {'Intervention'}")
print("-" * 110)
for rank, (_, row) in enumerate(top_10_features.iterrows(), 1):
    print(f"{rank:<5} {row['feature']:<35} {row['mean_abs_shap']:<12.4f} {row['interpretation']:<30} {row['intervention']}")
print("=" * 110)

In [ ]:
# Save explainability artifacts for audit trail / regulatory compliance
feature_importance.to_csv(os.path.join(OUTPUT_DIR, 'shap_feature_importance_full.csv'), index=False)
top_10_features.to_csv(os.path.join(OUTPUT_DIR, 'shap_feature_importance_clinical.csv'), index=False)

print(f"Saved: {os.path.join(OUTPUT_DIR, 'shap_feature_importance_full.csv')}")
print(f"Saved: {os.path.join(OUTPUT_DIR, 'shap_feature_importance_clinical.csv')}")

## 6d. TP / FN / FP Comparison with SHAP

Understanding **where and why** the model fails is critical for clinical deployment:

| Type | Definition | Question |
|------|-----------|----------|
| **True Positive (TP)** | High risk predicted, event happened | What drove the correct alert? |
| **False Negative (FN)** | Low risk predicted, event happened | Why did the model miss this one? |
| **False Positive (FP)** | High risk predicted, no event | What caused the false alarm? |

We explain each with both **force plots** and **natural language** to understand the model's reasoning.

---

<details>
<summary><strong>Hint 1 — TP/FN/FP masks</strong> (click to expand)</summary>

```python
tp_mask = (y_prob > threshold) & (y_test.values == 1)
fn_mask = (y_prob < 0.3) & (y_test.values == 1)
fp_mask = (y_prob > threshold) & (y_test.values == 0)
tp_idx = int(np.where(tp_mask)[0][0]) if tp_mask.any() else None
```
</details>

In [ ]:
# HOMEWORK TODO: Find TP, FN, FP instances from the test set predictions
#   TP: model predicted high risk AND event actually happened
#   FN: model predicted low risk BUT event actually happened (missed!)
#   FP: model predicted high risk BUT no event happened (false alarm)
threshold = 0.5

# TODO: Create boolean masks
#   tp_mask: y_prob > threshold AND y_test == 1
#   fn_mask: y_prob < 0.3 AND y_test == 1
#   fp_mask: y_prob > threshold AND y_test == 0
tp_mask = None
fn_mask = None
fp_mask = None

print(f"Test set: {len(y_test):,} instances")
print(f"  True Positives:  {tp_mask.sum() if tp_mask is not None else 'N/A'}")
print(f"  False Negatives: {fn_mask.sum() if fn_mask is not None else 'N/A'}")
print(f"  False Positives: {fp_mask.sum() if fp_mask is not None else 'N/A'}")

# TODO: Pick the first instance of each type using np.where
tp_idx = None
fn_idx = None
fp_idx = None

In [ ]:
# HOMEWORK TODO: Explain the TRUE POSITIVE patient
#   1. Get patient features: X_test.iloc[[tp_idx]]
#   2. Compute SHAP values: explainer.shap_values(features) -> get_shap_values()
#   3. Create force plot: shap.force_plot(exp_val, shap[0], features.iloc[0], matplotlib=True)
#   4. Generate clinical explanation using generate_patient_explanation()
#   5. Print with print_clinical_explanation()
if tp_idx is not None:
    tp_features = None  # TODO: X_test.iloc[[tp_idx]]
    tp_shap_raw = None  # TODO: explainer.shap_values(tp_features)
    tp_shap = None      # TODO: get_shap_values(tp_shap_raw)

    print("TRUE POSITIVE -- Model correctly identified high risk")
    print(f"Risk: {y_prob[tp_idx]:.1%}  |  Actual: HIGH-RISK EVENT\n")

    # TODO: Create force plot
    # shap.force_plot(float(exp_val), tp_shap[0], tp_features.iloc[0],
    #                 matplotlib=True, show=False)
    # plt.title(...)
    # plt.show()

    # TODO: Generate and print clinical explanation
    # tp_explanation = generate_patient_explanation(...)
    # print_clinical_explanation(tp_explanation, patient_id="TP Patient")

In [ ]:
# FALSE NEGATIVE: Model MISSED this patient
if fn_idx is not None:
    fn_features = X_test.iloc[[fn_idx]]
    fn_shap_raw = explainer.shap_values(fn_features)
    fn_shap = get_shap_values(fn_shap_raw)

    print("FALSE NEGATIVE \u2014 Model MISSED this patient")
    print(f"Risk: {y_prob[fn_idx]:.1%}  |  Actual: HIGH-RISK EVENT (missed!)\n")

    shap.force_plot(
        float(exp_val), fn_shap[0], fn_features.iloc[0],
        matplotlib=True, show=False
    )
    plt.title(f"FN Force Plot \u2014 Risk: {y_prob[fn_idx]:.1%} (MISSED)", fontsize=12, pad=20)
    plt.tight_layout()
    plt.show()

    fn_explanation = generate_patient_explanation(
        fn_features, fn_shap, feature_cols, y_prob[fn_idx]
    )
    print_clinical_explanation(fn_explanation, patient_id="FN Patient")

In [ ]:
# FALSE POSITIVE: False alarm
if fp_idx is not None:
    fp_features = X_test.iloc[[fp_idx]]
    fp_shap_raw = explainer.shap_values(fp_features)
    fp_shap = get_shap_values(fp_shap_raw)

    print("FALSE POSITIVE \u2014 False alarm")
    print(f"Risk: {y_prob[fp_idx]:.1%}  |  Actual: No event (false alarm)\n")

    shap.force_plot(
        float(exp_val), fp_shap[0], fp_features.iloc[0],
        matplotlib=True, show=False
    )
    plt.title(f"FP Force Plot \u2014 Risk: {y_prob[fp_idx]:.1%} (False Alarm)", fontsize=12, pad=20)
    plt.tight_layout()
    plt.show()

    fp_explanation = generate_patient_explanation(
        fp_features, fp_shap, feature_cols, y_prob[fp_idx]
    )
    print_clinical_explanation(fp_explanation, patient_id="FP Patient")

### Comparing TP vs FN \u2014 Why Did the Model Miss?

- **TP (True Positive)**: The model saw clear risk signals (e.g., elevated HbA1c, care gaps, recent ED visits) and correctly flagged the patient. The force plot shows strong red arrows pushing risk up.

- **FN (False Negative)**: The model missed this patient because the standard risk factors were absent or protective factors dominated. The force plot shows mostly blue arrows. The event may have been caused by a factor the model doesn't capture (social determinants, medication non-adherence, acute infection).

**This is exactly the analysis a clinical AI team would do during model validation** \u2014 understanding *why* the model fails helps identify what data or features to add next.

### FP \u2014 The Cost of False Alarms

The **FP (False Positive)** looks similar to a TP \u2014 the same features drive the prediction up. The model can't distinguish "at-risk patient who got lucky" from "at-risk patient who will have an event." This is inherent to probabilistic prediction and is why:
1. Clinical review is always needed before acting on predictions
2. The alert threshold must balance catch rate vs false alarm rate

## 6e. High-Risk vs Low-Risk Comparison

Comparing high-risk and low-risk patients validates the model's reasoning. If high-risk patients show sensible explanations while low-risk patients have the opposite pattern, we gain confidence the model learned meaningful clinical relationships.

In [ ]:
# Find low-risk patients
low_risk_all = np.where(y_prob < 0.15)[0]
low_risk_df = pd.DataFrame({
    'idx': low_risk_all,
    'patient_id': classifier_data.iloc[test_idx[low_risk_all]][patient_col].values
})
low_risk_sample = low_risk_df.drop_duplicates('patient_id').head(2)['idx'].values

print("LOW-RISK PATIENT EXAMPLES (for comparison)")
print("#" * 70)

for idx in low_risk_sample:
    patient_features = X_test.iloc[[idx]]
    patient_shap_raw = explainer.shap_values(patient_features)
    patient_shap = get_shap_values(patient_shap_raw)

    # Force plot
    shap.force_plot(
        float(exp_val), patient_shap[0], patient_features.iloc[0],
        matplotlib=True, show=False
    )
    plt.title(f"Low-Risk Force Plot \u2014 Risk: {y_prob[idx]:.1%}", fontsize=12, pad=20)
    plt.tight_layout()
    plt.show()

    explanation = generate_patient_explanation(
        patient_features, patient_shap, feature_cols, y_prob[idx], top_n=3
    )
    patient_id = classifier_data.iloc[test_idx[idx]][patient_col]
    print_clinical_explanation(explanation, patient_id=patient_id[:20] + "...")

**Key Contrast:**

| Factor | High-Risk Patients | Low-Risk Patients |
|--------|-------------------|-------------------|
| **Force plot** | Dominated by red (risk-increasing) arrows | Dominated by blue (protective) arrows |
| **HbA1c** | Often elevated or trending up | Well-controlled, stable |
| **Care gaps** | Frequent monitoring lapses | Regular follow-up visits |
| **SHAP budget** | Risk factors consume >60% of SHAP budget | Protective factors consume >60% |

This pattern validates the model learned clinically meaningful relationships \u2014 not spurious correlations.

## 6f. SHAP Limitations

SHAP is the best tool we have for tree-model explanations, but it is not perfect:

| Limitation | What It Means | Practical Impact |
|------------|---------------|------------------|
| **Feature independence assumption** | Shapley values assume features contribute independently | Correlated features (e.g., systolic & diastolic BP) split importance between them \u2014 neither gets full credit |
| **Correlation \u2260 separated credit** | Two correlated features share SHAP budget | A feature can appear "unimportant" simply because a correlated partner absorbed most of the attribution |
| **Explanations \u2260 causation** | SHAP shows what the *model* relies on, not what *causes* outcomes | High SHAP for `days_since_last_encounter` means the model uses it, not that skipping visits causes events |
| **Background data sensitivity** | TreeExplainer uses the training distribution as baseline | Explanations can shift if the patient population changes (e.g., different hospital, different demographics) |

**Bottom line:** SHAP explanations should inform clinical review, not replace it. Always pair SHAP outputs with domain expertise before acting on them.

---

# Part 7: Full End-to-End Demo

The pipeline's `explain_patient()` method wraps everything into a single call \u2014 the **production API** a colleague would use. It returns `contribution_pct` (no raw SHAP log-odds) for clinician-friendly output.

**Portability**: `ehr_pipeline.py` works standalone \u2014 import it anywhere:

```python
from ehr_pipeline import EHRPipeline, print_explanation

pipe = EHRPipeline(data_dir="path/to/csvs", model_path="path/to/model.pkl")
pipe.load_csvs().build_profiles()

report = pipe.explain_patient("patient-id", "2017-03-15")
print_explanation(report)
```

---

<details>
<summary><strong>Hint 1 — Pipeline API</strong> (click to expand)</summary>

`report = pipe.explain_patient(patient_id, date)` returns a dict with risk_score, risk_level, risk_factors, protective_factors. Print with `print_explanation(report)`.
</details>

In [ ]:
# HOMEWORK TODO: End-to-end pipeline demo
#   Use pipe.explain_patient() -- the production API -- on a sample patient.
#   This tests that your ehr_pipeline.py implementation works end-to-end.

demo_patient = list(pipe.profiles.keys())[10]
demo_date = "2015-06-15"

print("=" * 70)
print("END-TO-END PIPELINE DEMO")
print("=" * 70)
print(f"\nPatient: {demo_patient}")
print(f"Date:    {demo_date}")

# TODO: Call pipe.explain_patient(demo_patient, demo_date)
#   Then print the report with print_explanation() (from ehr_pipeline import)
report = None  # pipe.explain_patient(demo_patient, demo_date)

# TODO: print_explanation(report)

print("\n[Portability] ehr_pipeline.py works standalone --")
print("a colleague can import it and run predictions without this notebook.")

---

# Part 8: Deployment Assessment

Before deploying a model, we evaluate its performance and document its characteristics.

`evaluate_model()` computes:
- **PR-AUC**: Area under the Precision-Recall curve (most important for imbalanced data)
- **ROC-AUC**: Area under the ROC curve
- **Precision @ 80% recall**: What precision we get while catching 80% of true positives

In [ ]:
# Evaluate using pre-computed training data
print("Evaluating XGBoost model on pre-computed data...")
print("=" * 60)
metrics = pipe.evaluate_model(data=classifier_data)

In [ ]:
# Comparison table: progression from Week 2 to Week 3
print("\nMODEL PROGRESSION")
print("=" * 70)
print(f"{'Model':<40} {'ROC-AUC':>10} {'PR-AUC':>10}")
print("-" * 70)
print(f"{'Week 2: LR (6 features)':<40} {'~0.86':>10} {'~0.23':>10}")
print(f"{'Week 2: LR (22 features)':<40} {'~0.88':>10} {'~0.31':>10}")
print(f"{'Week 3: XGBoost (22 feat, tuned)':<40} {metrics['roc_auc']:>10.4f} {metrics['pr_auc']:>10.4f}")
print("=" * 70)
print(f"\nPR-AUC improvement: LR 6-feat -> XGBoost = ~{(metrics['pr_auc'] - 0.23) / 0.23 * 100:.0f}%")
print(f"Precision @ 80% recall: {metrics['precision_at_80_recall']:.1%}")

In [ ]:
# HOMEWORK TODO: Write a model card summarizing the model's characteristics.
#   Fill in the sections below using the metrics computed above.
#   This is a deployment readiness artifact required for regulatory compliance.

print("\n" + "=" * 70)
print("MODEL CARD: Diabetes High-Risk Event Predictor")
print("=" * 70)

# TODO: Fill in the model card sections
#   POPULATION: data source, patient count, instance count, positive rate
#   TASK: what the model predicts
#   MODEL: type, features, training approach
#   PERFORMANCE: ROC-AUC, PR-AUC, precision @ 80% recall
#   LIMITATIONS: at least 3 limitations
#   OPERATING THRESHOLD: recommended threshold with justification
#   FAIRNESS CONCERN: what hasn't been evaluated yet
#   GO/NO-GO RECOMMENDATION: conditional on what checks

print(f"""
POPULATION
  Source:        TODO
  Patients:      {classifier_data['patient_id'].nunique():,}
  Instances:     {len(classifier_data):,}
  Positive rate: {classifier_data[label_col].mean():.2%}

TASK
  TODO: Describe what the model predicts

MODEL
  Type:         TODO
  Features:     TODO
  Training:     TODO

PERFORMANCE (test set)
  ROC-AUC:                {metrics['roc_auc']}
  PR-AUC:                 {metrics['pr_auc']}
  Precision @ 80% recall: {metrics['precision_at_80_recall']}

LIMITATIONS
  1. TODO
  2. TODO
  3. TODO

OPERATING THRESHOLD
  Recommended: TODO
  Justification: TODO

FAIRNESS CONCERN
  TODO

GO/NO-GO RECOMMENDATION
  TODO
""")

---

## Part 9: Generalization -- New Patient Population

We proved the pipeline works end-to-end in Parts 1--8.
Now the real test: does the model generalize to **completely unseen patients**?

We load a separate 10K-patient dataset (different Synthea generation), run the full pipeline (load, clean, extract, profiles, features), then predict with the model trained on the original population.

> **Why this matters:** A model that only works on its training population is useless in deployment.
> Hospital systems constantly receive new patients -- your pipeline must handle them.

> **Note:** The cell below downloads the new data from Google Drive. If you already have the data locally or your paths differ, adjust `NEW_DATA_BASE`, `CSV_DIR`, and `NOTES_PATH` accordingly.

In [ ]:
# --- Download new patient population data from Google Drive ---
!pip install gdown -q
import gdown, os

NEW_DATA_BASE = os.path.join(REPO_PATH, 'data', 'week_3', 'new_data')
CSV_DIR = os.path.join(NEW_DATA_BASE, 'csv')
NOTES_PATH = os.path.join(NEW_DATA_BASE, 'notes_for_extraction_student.csv')

os.makedirs(CSV_DIR, exist_ok=True)

if not os.path.exists(os.path.join(CSV_DIR, 'patients.csv')):
    gdown.download_folder(
        'https://drive.google.com/drive/folders/19BzEc1OExgGmhlOBI58eGL1Yf0JAKf1F',
        output=CSV_DIR, quiet=False
    )
    print(f"CSVs downloaded to {CSV_DIR}")
else:
    print(f"CSVs already exist at {CSV_DIR}")

if not os.path.exists(NOTES_PATH):
    gdown.download(
        'https://drive.google.com/uc?id=1_JuB0jKbu3WqYvJVRzKJ8eUy7hJO56na',
        output=NOTES_PATH, quiet=False
    )
    print(f"Notes downloaded to {NOTES_PATH}")
else:
    print(f"Notes already exist at {NOTES_PATH}")

print(f"\nCSVs: {os.listdir(CSV_DIR)}")

In [ ]:
# --- New patient population ---
NEW_DATA_DIR = os.path.join(REPO_PATH, 'data', 'week_3', 'new_data', 'csv')
NEW_NOTES    = os.path.join(REPO_PATH, 'data', 'week_3', 'new_data', 'notes_for_extraction_student.csv')

# OCR config (same settings as Week 1 Session 3)
import json as _json
ocr_config = {
    "ocr_substitutions": {"0":"o","1":"l","3":"e","4":"a","5":"s","6":"g","7":"t","8":"b"},
    "valid_mixed_terms": ["a1c","hba1c","type1","type2","t1dm","t2dm","b12","d3",
                          "covid19","h1n1","mg","ml","kg","mcg","iu","24hr","12hr","2x","3x","4x"]
}
OCR_PATH = os.path.join(REPO_PATH, 'data', 'ocr_config.json')
with open(OCR_PATH, 'w') as f:
    _json.dump(ocr_config, f)
print(f"OCR config saved to {OCR_PATH}")

pipe_new = EHRPipeline(data_dir=NEW_DATA_DIR, model_path=MODEL_PATH, notes_file=NEW_NOTES, ocr_config=OCR_PATH)

# TODO: Run the full pipeline on new data (same steps as the original)
# Steps:
#   1. pipe_new.load_csvs()
#   2. pipe_new.clean_notes()     # HTML + OCR
#   3. pipe_new.extract_notes()   # NLP extraction
#   4. pipe_new.build_profiles()
#   5. new_features = pipe_new.build_feature_matrix(interval_days=30)
#
# We use interval_days=30 (monthly) -- enough for evaluation, ~4x faster.

# --- YOUR CODE HERE ---

raise NotImplementedError("Run the pipeline on new data")

# --- END YOUR CODE ---

print(f"Feature matrix: {new_features.shape}")

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve

target_col = 'will_have_high_risk_event_next_30d'

# Use the EXACT feature columns the model was trained on (same order)
X_new = new_features[pipe.feature_cols].values
y_new = new_features[target_col].values

# TODO: Predict on ALL new data using pipe.model, compute metrics
# Hint: pipe.model.predict_proba(X_new)[:, 1]

# --- YOUR CODE HERE ---

raise NotImplementedError("Predict on new data and compute metrics")

# --- END YOUR CODE ---

print("=== Generalization Metrics (New Population) ===")
print(f"  Samples:                {len(y_new):,}")
print(f"  Positive rate:          {y_new.mean():.3f}")
print(f"  ROC-AUC:                {roc_auc_new:.4f}")
print(f"  PR-AUC:                 {pr_auc_new:.4f}")
print(f"  Precision @ 80% recall: {prec_at_80:.4f}")

In [ ]:
# --- Side-by-side comparison ---
print("=" * 55)
print(f"{'Metric':<28} {'Original':>12} {'New Pop':>12}")
print("-" * 55)
print(f"{'ROC-AUC':<28} {metrics['roc_auc']:>12.4f} {roc_auc_new:>12.4f}")
print(f"{'PR-AUC':<28} {metrics['pr_auc']:>12.4f} {pr_auc_new:>12.4f}")
print(f"{'Precision @ 80% Recall':<28} {metrics.get('precision_at_80_recall', 0):>12.4f} {prec_at_80:>12.4f}")
print(f"{'Test Samples':<28} {metrics['test_samples']:>12,} {len(y_new):>12,}")
print(f"{'Positive Rate':<28} {metrics['positive_rate']:>12.3f} {y_new.mean():>12.3f}")
print("=" * 55)

roc_drop = metrics['roc_auc'] - roc_auc_new
print(f"\nROC-AUC change: {roc_drop:+.4f} ({'drop' if roc_drop > 0 else 'improvement'})")

### Interpreting Generalization Results

ROC-AUC held (~0.95 vs ~0.94) but **PR-AUC dropped by half** (0.88 → ~0.43). This is a generalization failure.

**Key observations:**
- The model can still *rank* patients (ROC-AUC), but its **precision is unacceptable** at every threshold.
- The positive rate halved (2.9% → 1.5%), meaning the new population has genuinely different disease dynamics.
- PR-AUC is prevalence-dependent: with fewer true positives, every false positive destroys precision.

> **HOMEWORK TODO:** In 3--4 sentences, explain why ROC-AUC held but PR-AUC collapsed.
> Would you deploy this model to the new population? What concrete steps would you take before deployment?

---

## Summary

### What You Accomplished

1. **Wired Week 1** OCR cleaning and note extraction into the pipeline
2. **Wired Week 2** patient profiles and 22-feature engineering
3. **Added prediction** using the Session 2 XGBoost model
4. **Global SHAP analysis** (Part 5):
   - Feature importance table (mean |SHAP|)
   - Beeswarm summary plot + bar plot
   - Feature dependence plots for top 4 features
5. **Individual explanations & clinical translation** (Part 6):
   - Force plots for high-risk patients
   - Natural language explanations with contribution_pct + SHAP values
   - Clinical interpretation reference with suggested interventions
   - TP/FN/FP comparison to understand model failures
   - High-risk vs low-risk patient contrast
   - SHAP limitations
6. **End-to-end demo** of the pipeline's production API (Part 7)
7. **Deployment assessment** with model card (Part 8)
8. **Generalization test** on a completely new patient population (Part 9)

### The Pipeline in One Diagram

```
Synthea CSVs  -->  load_csvs()  -->  clean_notes()  -->  extract_notes()
                                                              |
                                                              v
                                                       build_profiles()
                                                              |
                                                              v
                                        get_patient_features(patient_id, date)
                                                              |
                                                              v
                                        predict_patient()  /  explain_patient()
                                                              |
                                                              v
                                          SHAP Force Plots + Clinical Report
```

### Key Takeaways

- **Composition over reimplementation**: Every pipeline method imports and calls functions you built in previous weeks
- **NaN passthrough**: XGBoost handles missing values natively \u2014 no need for imputation
- **contribution_pct > raw SHAP**: Clinicians understand percentages, not log-odds
- **Force plots**: Visual "tug of war" between risk-increasing and protective factors
- **TP/FN/FP analysis**: Understanding model failures guides feature/data improvements
- **SHAP is not perfect**: Feature correlation, no causation, background sensitivity
- **Model is loaded, not trained**: The pipeline consumes the Session 2 artifact
- **Portability**: `ehr_pipeline.py` works standalone
- **Generalization is the ultimate test** — a model that only works on training data is useless in production \u2014 import it anywhere